# 09 — Ablaciones: qué aporta cada componente

Una ablación cambia un componente mientras mantiene fijos los demás para estudiar su aportación. Priorizaremos unas pocas preguntas del protocolo, con las mismas particiones, semillas y criterios de selección.

La lista de componentes es un menú de posibles contrastes, no una búsqueda exhaustiva ni una lista de experimentos terminados. La ablación Dense con/sin dropout ya está en 04; aquí se reutilizarán sus resultados.

> **Plan pendiente de implementación.** Este cuaderno contiene explicaciones y propuestas. No contiene código ejecutado ni resultados de entrenamiento.

## Ablaciones para aprender, no para perseguir el mejor resultado

La interpretación transversal de «don't look for performance» es que debemos poder explicar los experimentos. No significa dejar de medir ni relajar la evaluación. Para cada ablación: hipótesis, configuración de referencia, único cambio, presupuesto, semillas y resultado con variabilidad. Priorizar pocas comparaciones interpretables.

Ejemplos por vía: CNN con/sin aumento; LSTM con último estado frente a pooling con atención; encoder preentrenado congelado frente a fine-tuning; arquitectura idéntica con pesos aleatorios frente a preentrenados. Si se compara origen supervisado y autosupervisado con datos o presupuestos distintos, no atribuir toda la diferencia únicamente al objetivo de preentrenamiento. No es necesario ejecutar toda esta lista; se escogerá un subconjunto justificado. No usar el test para escoger la variante.

## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las 26 clases elegidas tras los controles, no todo el catálogo.

**Filtro y reutilización.** En 02 se aplica el mismo criterio visual fijo a train, validation y test; las fotos conservadas no cambian de partición. El test se reserva para evaluación predictiva, aunque haya pasado controles e inspección visual. Se guardarán resultados y predicciones para recalcular métricas sin repetir ajustes. Cualquier extracción o entrenamiento costoso nuevo requerirá una activación explícita; este cuaderno todavía no implementa ese flujo.

## Objetivos

- Registrar hipótesis, configuración, presupuesto y métricas antes del contraste.
- Priorizar normalización, aumentos, pesos de clase o inicialización cuando permitan responder una pregunta concreta.
- Separar el efecto observado del ruido entre semillas y de las diferencias de presupuesto.
- Calcular nuevas métricas e intervalos desde predicciones guardadas cuando no cambie el modelo.
- Reservar cualquier nuevo ajuste para una acción explícita y documentar las opciones que no se ejecuten.

## Pasos previstos

1. Revisar las hipótesis y recuperar los resultados existentes, incluido dropout de 04.
2. Elegir pocos contrastes con una sola diferencia identificable.
3. Medir el coste y registrar las configuraciones antes de ejecutarlas.
4. Comparar medias, desviaciones entre semillas e intervalos emparejados por grupos.
5. Interpretar efectos y límites sin elegir configuraciones con test.

## Entradas, salidas y material de clase

| | |
|---|---|
| **Entradas** | Configuraciones y resultados realmente completados en 04–08 |
| **Salidas** | Registro de contrastes, métricas y figuras en results/ |
| **Material de clase** | `PRDL_MLLB_FeedForward.pptx` (ablation studies, dropout), `MLLB_TransferLearning_Image_and-Text.pptx` (batch normalisation), `Notebooks_GuideLines_for_DeepLearnig_Report.pptx` |

## Decisiones pendientes

Medir primero tiempo y memoria en local. Priorizar pocos contrastes interpretables. Cualquier muestra por coste afectaría solo a train y necesitaría justificación, manteniendo validation/test.

## Comparación acordada: sin aumentos, suave y fuerte

Queremos comprobar si pequeñas variaciones ayudan a generalizar y si variaciones mayores aportan diversidad o destruyen información útil.

| Variante | Configuración real del lector |
|---|---|
| Referencia | `augment=False` |
| Suave | `augment=True, augmentation_profile='suave'` (brillo y saturación ±10 %) |
| Fuerte | `augment=True, augmentation_profile='fuerte'` (brillo y saturación ±50 %) |

Ambos perfiles tienen probabilidad 0,5 de reflejo horizontal. Solo se aplican en train, durante su lectura. Validation y test permanecen fijos. No se crean fotos nuevas en disco ni se modifica el manifiesto de 02.

Mantener la misma arquitectura, resolución, productos, normalización calculada con train sin aumentos, optimizador, semillas, límite de épocas y criterio de checkpoint. Comparar las semillas 42, 123 y 2026 emparejadas; medir el coste antes de ejecutar. La probabilidad del reflejo no cambia entre suave y fuerte.

Como los dos perfiles incluyen reflejo, sin/suave compara el conjunto de transformaciones; suave/fuerte estudia la intensidad conjunta de brillo y saturación. No permite separar el efecto individual de cada transformación.

Medir macro-F1 primaria, accuracy, resultados por clase e intervalos emparejados por grupos en validation, junto a tiempo y memoria. Registrar el perfil y sus valores en cada configuración e identificador de experimento; guardar predicciones y checkpoints para reutilizarlos. Elegir con validation, sin usar test para ajustar intensidades.

**Estado:** las dos opciones del lector están implementadas. Esta ablación de entrenamiento sigue pendiente; no hay resultados que demuestren qué opción ayuda más. No se programa ni inicia ningún entrenamiento desde este plan.
